# TESSERA embeddings of the RQ1 pilot points: rebuild and check

**Purpose.** Rebuilds the stored embedding set
`data/processed/embeddings/tessera_v1.1_2023_rq1_pilot/` from a fixed
point list and checks the stored files against it. Every experiment
reads this set.

**The point list is the input.**
[`data/tessera_rq1_pilot_points.csv`](../../data/tessera_rq1_pilot_points.csv)
(block, `px`, `py`: pixel centres in EPSG:32629) was written from the
stored blocks on 2026-11-13. The notebook that chose the points was
removed on 2026-10-20 and is not in the Git history, so the list cannot
be derived again from the source files; Section 1 shows which part of
it the current files explain.

**The embeddings can be read again from the list.** Section 2 reads
them from the GeoTessera store (a remote read of about 2 h, only with
`ALLOW_FETCH = True`) into a second folder and compares the two. The
stored blocks are kept when the arrays are identical, so their hashes,
and every file built from them, do not change.

**Status (2026-11-13): the full read was not done.** A run was stopped
after 14 of the 65 blocks; Section 2 compares those 14 with the stored
ones. The stored set is used as extracted on 2026-10-20, and the
manifest rows of its blocks are unchanged. Provenance:
[`data/README.md`](../../data/README.md).

In [ ]:
import sys
import warnings
from datetime import datetime
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src import manifest
from src.background import TESSERA_STORE, YEAR
from src.embeddings import PILOT_POINTS, compare_pilot_embeddings, fetch_pilot_embeddings, load_pilot_embeddings, pilot_points
from src.grid import pixel_id

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"
REFETCH = EMB_DIR / "tessera_v1.1_2023_rq1_pilot_refetch"
ALLOW_FETCH = False  # True reads every point again from the store (about 2 h)

points = pilot_points(ROOT)
print(f"{len(points)} points in {points.block.nunique()} blocks of 10 km")

## 1. What the point list holds

Each point is a 10 m pixel centre. The table says how many lie inside
a cleaned expert polygon, how many are the pixel of an *Acacia*
observation of the raw iNaturalist download, and how many are neither
(the negatives of the earlier design, which no current file
explains).

In [ ]:
pilot, emb = load_pilot_embeddings(PILOT)
expert = gpd.read_file(ROOT / "data/interim/expert_reference/expert_polygons_clean.gpkg").to_crs(32629)
raw = pd.read_csv(ROOT / "data/raw/citizen_science/inaturalist_acacia_coimbra_pilot.csv")
raw_pixels = set(pixel_id(gpd.GeoDataFrame(raw, geometry=gpd.points_from_xy(raw.longitude, raw.latitude), crs=4326)))

pts = gpd.GeoDataFrame(pilot, geometry=gpd.points_from_xy(pilot.x, pilot.y), crs=32629)
inside = gpd.sjoin(pts, expert[["geometry"]], predicate="within", how="inner").index.unique()
pilot["in_expert_polygon"] = pilot.index.isin(inside)
pilot["acacia_observation_pixel"] = pilot.pixel.isin(raw_pixels)
print(f"points without an embedding (NaN): {np.isnan(emb).any(axis=1).sum()}")
pd.crosstab(pilot.in_expert_polygon, pilot.acacia_observation_pixel, margins=True)

## 2. Read the embeddings again and compare

With `ALLOW_FETCH = True` every block of the list is read again into
the second folder (blocks already there are kept, so an interrupted
run can be resumed). The comparison is on the arrays: same points, same
values, NaN in the same places. File hashes are not compared, because
an `.npz` written at another time differs in its bytes.

In [ ]:
if ALLOW_FETCH:
    fetch_pilot_embeddings(ROOT, REFETCH)

check = compare_pilot_embeddings(ROOT, PILOT, REFETCH)
print(f"stored blocks holding exactly the listed points: {check.points_as_listed.sum()} of {len(check)}")
print(f"blocks read again: {check.refetched.sum()} of {len(check)} ({check.loc[check.refetched, 'n_points'].sum()} points)")
print(f"read again and identical to the stored block: {(check.same_points & check.same_embeddings).sum()}")
# blocks whose stored points differ from the list, or that were read again and differ
check[~check.points_as_listed | (check.refetched & ~(check.same_points & check.same_embeddings))]

## 3. Manifest

Written only when every block was read again and is identical. Each
stored block keeps its file and its build date; its row now names the
point list as input and records the date of the check.

In [ ]:
assert check.points_as_listed.all(), "the stored blocks do not hold the listed points"
if check.refetched.all():
    assert (check.same_points & check.same_embeddings).all(), "the store returned other values: nothing recorded"
    checked_on = datetime.fromtimestamp(max(f.stat().st_mtime for f in REFETCH.glob("block_*.npz"))).strftime("%Y-%m-%d")
    previous = manifest.read(ROOT)
    for block in check.block:
        file = PILOT / f"block_{block}.npz"
        manifest.record(
            ROOT, file,
            built_by="RQ1 pilot notebook removed on 2026-10-20; rebuilt and found identical by "
                     "notebooks/04_satellite/tessera_pilot_embeddings.ipynb",
            built_on=previous[file.relative_to(ROOT).as_posix()]["built_on"],
            inputs=[manifest.input_ref(ROOT, ROOT / PILOT_POINTS)],
            params={"tessera": TESSERA_STORE, "year": YEAR, "refetch_identical_on": checked_on},
        )
    print(f"{len(check)} manifest rows written; check of {checked_on}; folder hash {manifest.sha256_dir(PILOT)[:12]}")
else:
    print("not every block was read again: manifest left as it is")